# Abstract

This notebook logs the resampling of high-resolution spectrometer measurements of the ExoMars Mission Reference Samples for the purposes of simulating measurements with the Enfys flight model.

In this notebook we assume Enfys uses a modified InGaAs Mid-Wave Infrared (1.6 - 2.5 µm, 'MWIR') sensor, that is subject to thermal noise from the Enfys structure.

# Introduction

As discussed in [previous notebooks](./Simulating%20Spectral%20Sampling%20of%20the%20MICA%20Files%20with%20the%20Enfys%20IR%20Spectrometer%20for%20the%20ExoMars%20Rosalind%20Franklin%20Rover.ipynb), Enfys is an Infra-Red point-spectrometer for the ExoMars Rosalind Franklin rover.

In this notebook the objective is to produce 2 datasets of Enfys-like data, one without additive noise, and one with.

Analysis of the data is not within the scope of this notebook.

We simulate the expected performance of Enfys using modelled performance data for an InGaAs Short-Wave infrared sensor (0.9 - 1.7 µm) and a modified InGaAs Medium-Wave Infrared sensor for the wavelength range of 1.6 - 2.5 µm.
Included in the modelling is the noise expected for the sensor, as provided by the EXM-EN-MTX-ABU-0001_Enfys_Science_Traceability_Matrix_3-0 document (contact Enfys P.I. M. Gunn).


## Problem

[TODO overview of the ExoMars Mission Reference Samples]


# Method

[TODO description of how the data was collected and by what instruments, and how that data is aggregated and resampled here using SPTK.]

## Spectral Resampling
We take a simple approach to resampling, as follows. 

We neglect reflectance calibration uncertainty, and any systematic uncertainty when merging the data from the two separate photodiodes, and assume that the linear-variable filter (LVF) movement is perfectly calibrated such that a measurement at centre-wavelength $\lambda_{cwl}$ can be requested with arbitrary precision.

We assume an observation of $\lambda_{cwl}$ has a Cauchy transmission profile. We use the latest expectations of the spectral resolving power of the short-wave infrared (SWIR) and mid-wave infrared (MWIR) LVFs, that we have linearly interpolated from ~30 measurements from 0.9 - 2.6 µm to arbitrary values of $\lambda$ in the SWIR and MWIR ranges, to give the vector $\Gamma[\lambda]$. The Cauchy transmission profile for a given $\lambda_{cwl}$ has a full-width-at-half-maximum of $\Delta\lambda = \lambda_{cwl}/\Gamma[\lambda_{cwl}]$.

This data has been digitsed from the plot reported in the `Enfys schedule review 20250807.ppt`, by Matt Gunn.

In [ ]:
import pandas as pd

enfys_swir_respow = pd.read_csv('../data/instruments/enfys_swir_respow.csv', index_col=0)
enfys_mwir_respow = pd.read_csv('../data/instruments/enfys_mwir_respow.csv', index_col=0)
g=enfys_swir_respow.plot(label='SWIR', marker='o')
enfys_mwir_respow.plot(marker='o',label='MWIR', xlabel='Wavelength (nm)', ylabel=r'$\lambda/\Delta\lambda$ Resolving Power',ax=g)
g.legend(['SWIR', 'MWIR'])
title = g.set_title('Enfys SWIR & MWIR Spectral Resolving Power $\Gamma[\lambda]$')

**Figure 1** Enfys SWIR & MWIR Spectral Resolving Power $\Gamma[\lambda]$, extracted from `Enfys schedule review 20250807.ppt`, M. Gunn.


The Cauchy transmission profile $T(\lambda|\lambda_{cwl})$ is given by

$$
T(\lambda|\lambda_{cwl}) = \frac{\gamma^2}{(\lambda - \lambda_{cwl})^2 + \gamma^2}
$$

where $\gamma = \Delta\lambda/2 = (\lambda_{cwl}/\Gamma[\lambda_{cwl})/2$.

The shape of the Cauchy profile is illustrated here.

In [ ]:
from sptk.instrument import Instrument
import sptk.config as cfg
from matplotlib import pyplot as plt

cfg.update_sample_res('wvl_min', 800)
cfg.update_sample_res('wvl_max', 2600)

exmpl_cauchy = Instrument.build_cauchy_filter(1600, 1600/100)
plt.plot(cfg.WVLS, exmpl_cauchy.squeeze())
plt.xlim(1500, 1700)
plt.xlabel('Wavelength (nm)')
plt.ylabel('Relative Transmittance')
title = plt.title('Example Cauchy Filter at 1600 nm with Spectral Resolving Power $\Gamma$ = 100')

**Figure 2** Example Cauchy Filter transmission profile, at 1600 nm with Spectral Resolving Power $\Gamma$ = 100.


We assume that a high-resolution laboratory spectrum approximates the continuous function $R(\lambda)$, such that the calibrated reflectance at $\lambda_{cwl}$ has the expected value $R[\lambda_{cwl}]$ of:

$$
R[\lambda_{cwl}] = \frac{\int R(\lambda)T(\lambda|\lambda_{cwl}) d\lambda }{\int T(\lambda|\lambda_{cwl}) d\lambda}
$$

We use this equation to compute the Enfys-sampled reflectance spectra.

## Adding Noise

To simulate noise we add $\varepsilon$ to a reflectance measurement, an additive noise term drawn from the Gaussian distribution with standard deviation $\sigma$, that we define relative to the specified Signal-to-Noise Ratio ($\text{SNR}$),

$$\sigma = \frac{S}{\text{SNR}}$$

where $S$ is the signal.

The supplied Signal-to-Noise Ratio vector has been defined under the assumption of a 30% albedo spectrally uniform reflector. 

The expected spectral SNR of the SWIR and MWIR sensors has been measured and processed by P.I. M. Gunn, and reported in the `EXM-EN-MTX-ABU-0001_Enfys_Science_Traceability_Matrix_3-0` spreadsheet.

We read in the SNR file, and get the SNR data for the InAs MWIR sensor and the InGaAs SWIR sensor,

In [ ]:
enfys_snr = pd.read_csv('../data/instruments/enfys_snr.csv', index_col=0)
ingaas_mwir_snr = enfys_snr['InGaAs MWIR SNR']
ingaas_swir_snr = enfys_snr['InGaAs SWIR SNR']

g = ingaas_swir_snr.plot(grid=True, label='Nominal InGaAs SWIR SNR')
ingaas_mwir_snr.plot(logy=True, grid=True, label='Nominal InGaAs MWIR SNR')
g.set_xlabel('Wavelength (nm)')
g.set_ylabel('SNR')
g.legend()
title = g.set_title('Spectral SNR of InGaAs SWIR and InGaAs MWIR Sensors')

**Figure 3** Spectral Signal-to-Noise Ratio for InGaAs SWIR and InGaAs MWIR Sensors, extracted from `EXM-EN-MTX-ABU-0001_Enfys_Science_Traceability_Matrix_3-0` spreadsheet, M. Gunn.


We assume that this SNR vector is modulated by the reflectance of the mineral of interest, such that $\sigma$ as a function of the spectral Signal-to-Noise Ratio $\text{SNR}[\lambda]$ is:

$$\sigma[\lambda] = \frac{R[\lambda]}{\text{SNR}[\lambda]}$$


For each observation $R[\lambda_{cwl}]$ we draw $n$-repeat noisy samples $\tilde{R}_i[\lambda_{cwl}]$, such that

$$
\tilde{R}_i[\lambda_{cwl}] = R[\lambda_{cwl}] + \varepsilon_i[\lambda_{cwl}]
$$

with 

$$
\varepsilon_i[\lambda_{cwl}] \sim \mathcal{N}\left(0, \sigma[\lambda_{cwl}] \right)
$$

This gives an indication of the expected noise of a calibrated *Enfys* acquired spectrum.


## Analysis

First we produce plots of the resampled spectra, and single instances of noisy spectra, and visually inspect these to consider the ability of this Enfys configuration to resolve diagnostic spectral features against the noise.

### Spectral Angle

We then take a single noisy sample and evaluate the Spectral Angle between the noisy sample and each noise-free entry of the MICA files spectral database that we are sampling. We repeat this for each noisy sample, to give the mean Spectral Angle and it's uncertainty, to determine the ability to dicriminate between similar mineral reflectance spectra.

The Spectral Angle is defined[^kruse1993] between the noisy spectrum $\tilde{R}_i[\lambda]$ of material $i$ in the set of MICA file materials ($\mathcal{M}$) and the clean spectrum $R_j[\lambda]$, where $j \in \mathcal{M}$.

The Spectral Angle $\theta$ is:

$$\theta(i,j) = \arccos \left( \frac{\sum_{\lambda} \tilde{R}_i[\lambda] {R}_j[\lambda]}{ \left(\sum_{\lambda} \tilde{R}_i[\lambda]^2 \right)^{1/2} \left(\sum_{\lambda} R_j[\lambda]^2 \right)^{1/2} }  \right) $$

This closed form expression can be formulated in a way such that $\theta$ can be computed for all $\mathcal{M}$ for a given material $i$ in parallel, as well as multiple noisy repeat samples of $i$.

[^kruse1993]: F.A. Kruse, A.B. Lefkoff, J.W. Boardman, K.B. Heidebrecht, A.T. Shapiro, P.J. Barloon, A.F.H. Goetz, The spectral image processing system (SIPS)—interactive visualization and analysis of imaging spectrometer data, Remote Sensing of Environment, Volume 44, Issues 2–3, 1993, Pages 145-163, ISSN 0034-4257, https://doi.org/10.1016/0034-4257(93)90013-N

## Notebook and Environment Setup

Here we include the required code blocks for setting up this notebook for this study.

In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'svg' # note - requires Inkscape to be installed, and to be accessible from the terminal
# on macos, add inkscape to the path with `sudo ln -s /Applications/Inkscape.app/Contents/MacOS/inkscape /usr/local/bin/inkscape`
%matplotlib inline

We will be using the Spectral Parameters Toolkit. First we import the required modules.

In [ ]:
import sptk.config as cfg
from sptk.material_collection import MaterialCollection
from sptk.instrument import Instrument
from sptk.observation import Observation
from sptk.spectral_library_analyser import SpectralLibraryAnalyser as sla

We set our simulation resolution range to extend the range of Enfys, of 0.9 - 2.5 µm, such that the long tails of the Cauchy distribution of the linear-variable-filter profiles can be accommodated during sampling. We leave our spectral resolution at 1 nm for all $\lambda$.

In [ ]:
cfg.update_sample_res('wvl_min', 800)
cfg.update_sample_res('wvl_max', 2600)

Finally we set the project name.

In [ ]:
PROJECT_NAME = 'enfys_mission_reference_samples'

# Data

TODO - better write up of this.

Data was collected at Aber. Uni. by Enfys team (PG, CC, LP).

Here we use data from the visible-to near infrared spectrometer.

First we prepare and SPTK compatible spectral library from the data, and then we ingest this library into the notebook via the Material Collection class.

## Preparing the Dataset

The data is held adjacent to this directory

In [ ]:
from pathlib import Path
raw_data_dir = '../../enfys_data/RS-3500_spectrometer/'
raw_data_files = sorted(list(Path(raw_data_dir).glob('*.sed')))

Let's try to read one of these files into the notebook, using pandas.

In [ ]:
import pandas as pd
from sptk.material_collection import header_template
from sptk.material_collection import MaterialCollectionBuilder

In [ ]:
library = 'EXM_MRS'

In [ ]:
def create_material_entry(raw_data_file: Path, spectral_library: str) -> tuple[dict, pd.Series]:
    """
    Create a material entry from a raw .sed file from the ExoMars Mission Reference Samples dataset.

    :param raw_data_file: path to the raw .sed file
    :type raw_data_file: Path
    :return: header dictionary and series of reflectance data
    :rtype: tuple(dict, pd.Series)
    """
    # read the raw text file
    # it's not so efficient to read each file twice, but we don't have 1000's to read, so it's ok
    raw_hdr = pd.read_csv(raw_data_file, header=None, nrows=26, sep=': ', index_col=0, engine='python')
    raw_data = pd.read_csv(raw_data_file, header=26, sep='\t', index_col=0, engine='python')

    # extract the reflectance data
    raw_refl = raw_data['Reflect. %']

    # write this information to a Material Collection compatible .csv file.
    # this should really be a class/function of the MaterialCollection module, in the same way we have an InstrumentBuilder class of the Instrument module.
    header = header_template()

    # gathering header metadata
    sample_id = '_'.join(raw_data_file.stem.split('_')[1:])
    species = '_'.join(raw_data_file.stem.split('_')[1:-1])
    # we will use the species name as the group and subgroup
    group = species
    subgroup = species
    sample_description = raw_hdr.loc['Comment'].values[0]
    date_added = raw_hdr.loc['Date'].values[0]
    database_of_origin = 'ExoMars Mission Reference Samples'
    other_information = f"Acquired with {raw_hdr.loc['Instrument']}; {raw_hdr.loc['Integration'].values[0]} integration times; {raw_hdr.loc['Averages'].values[0]} scans averaged"
    resolution = "2.8nm@700nm 8nm@1500nm 6nm@2100nm"
    # putting the information in the header dictionary
    header['Sample ID'] = sample_id
    header['Species'] = species
    header['Subgroup'] = subgroup
    header['Group'] = group
    header['Library'] = library
    header['Sample Description'] = sample_description
    header['Date Added'] = date_added
    header['Database of Origin'] = database_of_origin
    header['Resolution'] = resolution
    header['Other Information'] = other_information

    raw_refl.index.name = 'Wavelength'
    raw_refl.name = 'Response'
    raw_refl = raw_refl / 100  # convert from % to a fraction

    return header, raw_refl

In [ ]:
exm_mrs_lib = MaterialCollectionBuilder(library)

In [ ]:
for raw_data_file in raw_data_files:
    header, data = create_material_entry(raw_data_file, library)
    sample_id = header['Sample ID']
    print(f"Adding {sample_id} to {library} library")
    exm_mrs_lib.add_material(header, data)

Now we need to construct a material set for categorising these.

In [ ]:
exm_mrs_lib_map = exm_mrs_lib.map_spectral_library()

In [ ]:
exm_mrs_lib_map.keys()

In [ ]:
EXM_MRS_LIB = {}

samples_list = []
for key in exm_mrs_lib_map.keys():
    samples_list.append(key)
# move the 'Standard' and 'Standard_retest' to a separate 'standards' list
EXM_MRS_LIB['standards'] = [(s, '*') for s in samples_list if 'Standard' in s]
EXM_MRS_LIB['samples'] = [(s, '*') for s in samples_list if 'Standard' not in s]

In [ ]:
EXM_MRS_LIB

## Material Collection Construction

We use the ```sptk.material_collection``` module to parse the spectral library:

In [ ]:
matcol = MaterialCollection(
        EXM_MRS_LIB,
        'EXM_MRS', 
        PROJECT_NAME, 
        load_existing=False,
        balance_classes=False,
        random_bias_seed=None,
        allow_out_of_bounds=True,
        plot_profiles=False,
        export_df=True)

We can use the ```material_collection``` plotting routines to visualise the library.

In [ ]:
ax = matcol.plot_profiles(stacked=False, scope='species', groupby='Sample ID')

**Figure 4** High resolution MICA Files spectral library, arranged by category and grouped by mineral species.

We can see that not all of the entries cover the complete 0.9 - 3.1 µm range of *Enfys*, so in future work we should look to replace these MICA files laboratory type spectra with spectra that all span the entire range.

We can show band locations more clearly after continuum removal. We can use the `SpectralLibraryAnalyser.remove_continuum()` function to produce a duplicate of the `MaterialCollection` but with the data continuum-removed.

In [ ]:
from sptk.spectral_library_analyser import SpectralLibraryAnalyser as sla

matcol_sla = sla(matcol)
matcol_cr = matcol_sla.remove_continuum()

We can plot the profiles to illustrate the continuum removal.

In [ ]:
axes = matcol_cr.plot_profiles(stacked=True, scope='species', groupby='Sample ID')

**Figure 5** Continuum-removed high resolution MICA Files spectral library, grouped by Category.

We can illustrate the band locations more clearly with the 'waterfall' visualisation:

In [ ]:
axes = matcol_cr.plot_profiles(waterfall=True, scope='species', groupby='Sample ID')

**Figure 6** Continuum-removed waterfall plot of high resolution MICA Files spectral library, grouped by Category.

We can see that the waterfall plot represents the band depth information of the `MaterialCollection` in a compact format. Scanning vertically allows for quick identification of common band locations, their depths, widths, and asymmetries.

## Simulating the Enfys Spectral Response

To simulate Enfys, rather than providing a file with the expected spectral response of each position of the Linear-Variable Filter (LVF, above), we specify the spectral range and resolution and sampling regime, and generate a set of central-wavelengths and bandwidths. We do this with the ```InstrumentBuilder``` class of the ```sptk.instrument``` module.

In [ ]:
from sptk.instrument import InstrumentBuilder

### Signal-to-Noise Ratio

Previously we loaded the spectral SNR, as extracted from the current version of the Enfys STM. Here we merge the SWIR and MWIR SNR datasets, and prepare the data for passing to the `sptk.Instrument` constructor.

We have adapted the InstrumentBuilder function to accept a pandas Series mapping wavelength to SNR. The InstrumentBuilder then performs interpolation to the simulated wavelength range.

We splice the SWIR and MWIR data together, using the higher value SNR where the wavelength ranges overlap:

In [ ]:
# concatanate the inas_mwir_snr and the ingaas_swir, using the highest SNR for overlapping wavelength values
enfys_ingaas_mwir_snr = ingaas_swir_snr.combine_first(ingaas_mwir_snr)

We also convert the wavelengths from µm to nm:

In [ ]:
enfys_ingaas_mwir_snr.index = enfys_ingaas_mwir_snr.index * 1000

for computational convenience, we should set values of SNR<=3 to NaN, to interpret these as 'bad bands'

In [ ]:
# where SNR <=1 to NaN, to interpret these as 'bad bands'
import numpy as np
enfys_ingaas_mwir_snr = enfys_ingaas_mwir_snr.where(enfys_ingaas_mwir_snr > 3, other=np.nan)

In [ ]:
enfys_ingaas_mwir_snr.index.name = 'wavelength'

The continuous spectral SNR for the complete instrument is:

In [ ]:
g = enfys_ingaas_mwir_snr.plot(logy=True)
# set the x and y axes
g.set_xlabel('Wavelength (nm)')
g.set_ylabel('SNR')
title = g.set_title('Enfys InGaAs SWIR & InGaAs MWIR Merged Sensor SNR')

**Figure 7** SWIR & MWIR merged spectral Signal-to-Noise Ratio input data, extracted from the current Enfys Science Traceability Matrix, maintained by M. Gunn.

### Spectral Resolving Power

Previously we loaded in the latest data for modelling the Spectral Resolving Power for the SWIR and MWIR LVFs. Here we merge the dataset, and prepare it to be passed to the `sptk.Instrument` module.

We merge the datasets, using the SWIR values where the sensor range overlaps.

In [ ]:
# get the max wavelength where ingaas_swir_snr is not nan
ingaas_swir_snr = ingaas_swir_snr[~np.isnan(ingaas_swir_snr)]
max_swir_wavelength = ingaas_swir_snr.index[-1] * 1000 # convert from µm to nm
max_swir_wavelength
# get the enfys_swir_respow data where the wavelength is less than the max_swir_wavelength
enfys_swir_respow = enfys_swir_respow[enfys_swir_respow.index < max_swir_wavelength]
enfys_swir_respow
# get the enfys_mwir_respow where the wavelength is greater than the max_swir_wavelength
enfys_mwir_respow = enfys_mwir_respow[enfys_mwir_respow.index > max_swir_wavelength]
enfys_mwir_respow
# now merge the datasets
enfys_respow = pd.concat([enfys_swir_respow, enfys_mwir_respow])

The complete instrument Spectral Resolving Power is:

In [ ]:
g=enfys_respow.plot(marker='o',xlabel='Wavelength (nm)', ylabel=r'$\lambda/\Delta\lambda$ Spectral Resolving Power')
g.legend(['SWIR & MWIR'])
title = g.set_title('Enfys SWIR & MWIR merged Spectral Resolving Power $\Gamma[\lambda]$')

**Figure 8** SWIR and MWIR merged Spectral Resolving Power data, as extracted from the `Enfys schedule review 20250807.ppt`, M. Gunn.

We pass this data to the InstrumentBuilder function.

### Instrument Build

Now we will pass this to the InstrumentBuilder, defining the spectral range as $\lambda \in [900, 3100]$ nm.

In [ ]:
enfys_builder = InstrumentBuilder(
    instrument_name='enfys_ingaas_mwir',
    instrument_type='lvf',
    sampling='nyquist',
    resolution = enfys_respow,
    spectral_range=[900, 2501],
    snr = enfys_ingaas_mwir_snr)

In [ ]:
from sptk.instrument import Instrument

Now we build the transmission profiles for each LVF position, according to the list of CWLs and FWHMs generated above. These are accessed by reading the ```.csv``` file produced by the ```InstrumentBuilder``` procedure.

When generating the profiles, we turn the CWL and FWHM into a profile by assuming a distribution function. For Enfys we use the Cauchy distribution, under the guidance of the Enfys build team, which behaves similarly to a Gaussian, but with wider 'wings'. It is parameterised by the full-width-at-half-maximum, as the distribution strictly has no finite variance. We have implemented a ```build_cauchy_filter``` method for the ```Instrument``` class.

In [ ]:
enfys = Instrument(
            name = 'enfys_ingaas_mwir', # filename of the instrument information, held in /software/data/instruments/            
            project_name = matcol.project_name, # the project directory name, hosting outputs in the /spectral_parameter_studies/ directory
            shape = 'cauchy',
            load_existing=False, # if True, load existing instrument data from the project directory
            plot_profiles=False, # if true, plot the transmission profiles during construction
            export_df=True) # if True, export the instrument transmission profiles to the project directory

In [ ]:
fig ,ax = enfys.plot_instrument_characteristics()

In [ ]:
enfys.cwls()

In [ ]:
enfys.fwhms()['S119']

**Figure 9** Simulated spectral response and resolution of *Enfys*, based on preliminary expectations of the spectral and radiometric response.

The data shown in figure 9 approximates the expected performance of *Enfys*. This data represents the current component level understanding of the filter and sensor performance of the built instrument.

# Results

## Sampling with Enfys

We now make an ```Observation``` object by sampling the MICA files ```MaterialCollection``` with the Enfys ```Instrument```.

In [ ]:
from sptk.observation import Observation

In [ ]:
obs = Observation(
    material_collection=matcol,
    instrument = enfys,
    load_existing=False,
    plot_profiles=False,
    export_df=True)

We can plot the sampled spectra against the input high-spectra below:

In [ ]:
# axes = obs.plot_profiles(scope='categories', groupby='Subgroup', stacked=True, hires_under=True)

axes = obs.plot_profiles(stacked=False, scope='species', groupby='Sample ID', hires_under=True)

**Figure 10** The MICA files as sampled by *Enfys*, with the original high-resolution spectra plotted underneath each.

In [ ]:
axes = obs.plot_profiles(scope='8_Mudstone', groupby='Sample ID', stacked=False, hires_under=True)

**Figure 11** Deviation of the *Enfys* sampled spectrum from the high-resolution input spectrum for Illite.

We note that there is an effect by which peaks and troughs in the spectra are reduced. This is due to the long tails of the Cauchy filter profiles. Despite the reduction of absolute band depth, the features of Illite (fig. 8) are still resolvable.

We can produce a continuum-removed waterfall plot to check that the major bands are resolved.

In [ ]:
from sptk.spectral_library_analyser import SpectralLibraryAnalyser as sla

obs_cr = sla(obs).remove_continuum()

In [ ]:
obs_cr.main_df

In [ ]:
# axes = obs_cr.plot_profiles(waterfall=True, scope='all', groupby='Category')

axes = obs_cr.plot_profiles(waterfall=True, scope='species', groupby='Sample ID')

**Figure 12** Continuum-Removed waterfall plot of the MICA files as sampled by *Enfys*.

Comparing Figure 12 to Figure 6, we can see that the weaker sub-2 µm bands are less well resolved.

We can save this data

In [ ]:
obs_cr.export_main_df()

## Adding Synthetic Noise

We have the capability to add synthetic noise to the dataset, by redrawing a sample from the dataset with noise added according to the signal-to-noise ratio specified for the given channel.

We can optionally redraw multiple noisy samples for each entry, to give an indication of the statistical uncertainty, or for example draw a single noisy sample, to produce a single-observation mission-realistic dataset.

First, let's draw a single noisy entry:

In [ ]:
noise_obs = obs.add_noise(1, seed=0)

In [ ]:
axes = obs.plot_profiles(scope='8_Mudstone', groupby='Sample ID', stacked=False, hires_under=True, ci=False)

**Figure 13** Noisy resampled example spectrum for Illite, showing 1 noisy sample according to the spectral signal-to-noise ratio illustrated in figure 9.B.

We can illustrate this in greater deal by plotting each entry on a separate figure, with the y-axis clipped to the spectral range of entry. Note that for samples with a small spectral range (i.e. samples that are spectrally flat), this stretching will appear to amplify noise, so the y-axis range must be considered when assessing the noise.

In [ ]:
axes = obs.plot_profiles(scope='samples', groupby='Species', stacked=False, hires_under=True, ci=False)

**Figure 14** Detailled view of each entry in the spectral library, showing the scale of noise introduced, relative to the features of each spectrum, and the reflectance range.

Qualitatively, these plots illustrate that *Enfys* does not significantly intrude on the diagnostic features of these type specimen of minerals identified on the Mars surface. In some cases, we can see that doublet features are lost, but this is due to the spectral resolving power, rather than the additive noise. We will assess this in the discussion section.

We can export the dataset:

In [ ]:
obs.export_main_df()

## Continuum Removal

We can apply continuum removal, even to the noisy data.

In [ ]:
from sptk.spectral_library_analyser import SpectralLibraryAnalyser as sla

cr_tool = sla(obs)
cr_obs = cr_tool.remove_continuum()

In [ ]:
axes = cr_obs.plot_profiles(scope='species', groupby='Sample ID', stacked=True, hires_under=False, ci=False)

**Figure 15** Continuum-Removed entries of the MICA Files, as sampled by *Enfys*, including synthetic noise.

Again, we can export this:

In [ ]:
cr_obs.export_main_df()

# Sample Analysis

In this section we look at each sample set in turn.

## 5. Granby

First we access the Granby samples

![#5 Granby - Context Image](../context_images/granby_context.JPG)

**#5 Granby - Context Image**

![#5 Granby - Spectrometer Sample Point](../context_images/granby_spot.jpeg)

**#5 Granby - Spectrometer Sample Point**

In [ ]:
obs.get_subset_df(category='samples').Group.unique()

In [ ]:
sample = '5_Granby'

In [ ]:
obs.get_subset_df(group=sample)

First let's plot the reflectance data without stacking.

In [ ]:
axes = obs.plot_profiles(scope=sample, groupby='Sample ID', stacked=False, hires_under=False, ci=False)

Now let's look at it stacked:

In [ ]:
axes = obs.plot_profiles(scope=sample, groupby='Sample ID', stacked=True, hires_under=False, ci=False)

Now let's look at the continuum removed data.

In [ ]:
axes = obs_cr.plot_profiles(scope=sample, groupby='Sample ID', stacked=False, hires_under=False, ci=False)

In [ ]:
axes = obs_cr.plot_profiles(scope=sample, groupby='Sample ID', stacked=True, hires_under=False, ci=False)

We see a broad and deep absorption centred ~1.1 µm, and narrow and shallow absorption at ~1.43 µm, and a deep and assymetric absorption at ~1.92 µm. there is another absorption at ~2.30 µm.

# Comparison to MICA Files

In [ ]:
from sptk.material_sets import MICA_SET

mica_mc = MaterialCollection(
        MICA_SET,
        'MICA_new_dirtree', 
        PROJECT_NAME, 
        load_existing=False,
        balance_classes=False,
        random_bias_seed=None,
        allow_out_of_bounds=True,
        plot_profiles=False,
        export_df=True)

In [ ]:
mica_ob = Observation(
    material_collection=mica_mc,
    instrument = enfys,
    load_existing=False,
    plot_profiles=False,
    export_df=True)

In [ ]:
mica_ob.main_df

In [ ]:
obs.main_df

In [ ]:
def spectral_angle(data_a: np.ndarray, data_b: np.ndarray):
    """Compute the Spectral Angle between dataset a and dataset b,
    where a and b have the same dimensions

    :param data_a: Observation Data
    :type data_a: np.ndarray
    :param data_b: Comparison Data
    :type data_b: np.ndarray
    :return: Spectral Angle array
    :rtype: np.ndarray
    """    
    # check dimensions of a and b match
    # if data_a.shape != data_b.shape:
    #     raise ValueError(f"Arrays must have the same dimensions. Got {data_a.shape} and {data_b.shape}")

    # compute spectal angle
    numerator = np.inner(data_a, data_b) #TODO replace inner with nansum
    denominator = np.outer(np.sqrt(np.sum(data_a**2, axis=1)), np.sqrt(np.sum(data_b**2, axis=1)))
    x = numerator / denominator
    # check range of x is in [-1, 1]
    x = np.clip(x, -1, 1)
    sa = np.arccos(x)
    # convert from radians to degrees
    sa = np.degrees(sa)
    return sa

def compute_spectral_angle(tgt_obs: Observation, ref_obs: Observation, 
                           material_name: str):
    """Compute the spectral angle between a given material in a target set of
    observed data against a reference set of data.

    :param tgt_obs: _description_
    :type tgt_obs: _type_
    :param ref_obs: _description_
    :type ref_obs: _type_
    :param material_name: _description_
    :type material_name: _type_
    :return: _description_
    :rtype: _type_
    """    
    # get Target data
    tgt_data = tgt_obs.get_refl_df(root_data_id=material_name).to_numpy()
    bad_bands = ~np.isfinite(tgt_data[0,:])
    tgt_data = np.where(bad_bands, 0, tgt_data) # set all nan values to 0

    # get Reference data
    ref_data = ref_obs.main_df[ref_obs.wvls].to_numpy()
    ref_data = np.where(bad_bands, 0, ref_data) # 0 the clean data where the noisy data is NaN
    ref_data = np.where(~np.isfinite(ref_data), 0, ref_data) # 0 the clean data where the clean data is NaN

    sa_noisy = spectral_angle(tgt_data, ref_data)

    # get baseline Spectral Angle
    baseline_data = tgt_obs.noiseless_df.loc[material_name][tgt_obs.wvls].to_numpy() 
    baseline_data = np.expand_dims(baseline_data, axis=0).astype(np.float64)  
    baseline_data = np.where(bad_bands, 0, baseline_data) # 0 the clean data where the noisy data is NaN
    baseline_data = np.where(~np.isfinite(baseline_data), 0, baseline_data) # 0 the clean data where the clean data is NaN

    sa_baseline = spectral_angle(baseline_data, ref_data)

    # package in DF
    sa_df = pd.DataFrame(sa_noisy, index=tgt_obs.get_refl_df(root_data_id=material_name).index, columns=ref_obs.main_df.index)
    sa_bl = pd.Series(sa_baseline.squeeze(), index=ref_obs.main_df.index)
    # # set sa_bl for index of material_name to 0
    # sa_bl.loc[material_name] = 0.0

    # # order DF columns in baseline value order
    sa_df = sa_df.loc[:, sa_bl.sort_values().index]
    sa_bl = sa_bl.loc[sa_bl.sort_values().index]

    return sa_df, sa_bl

Checking MICA files against itself for numerical correctness:

In [ ]:
mica_ob_noise = mica_ob.add_noise(1, seed=0)

In [ ]:
mica_test_material = mica_ob_noise['Root Data ID'].unique()[-1]

In [ ]:
mica_test_material

In [ ]:
sa_df, sa_bl = compute_spectral_angle(mica_ob, mica_ob, mica_test_material)

In [ ]:
sa_df.mean(axis=0).sort_values()

Indeed the Mg-sulphate test material is recovered as the best Spectral Angle match.

Let's now plot the results.

In [ ]:
# alternatively, we can plot the Spectal Angle score against the entry
def plot_mean_spectral_angle(sa_df, material_name, top_n: int=10, ax: plt.Axes=None):
    if ax is None:
        fig, ax = plt.subplots()
    else:
        fig = ax.figure

    top_n_sa_df = sa_df.iloc[:,:top_n]

    ax.plot(top_n_sa_df.mean(axis=0), marker='o', label='Ave. ± Std. Dev.')
    ax.legend(loc='upper left')
    # set ylimits
    # ax.set_ylim(0, 1)
    # add error bars if there are repeat noisy measurements
    if sa_df.shape[0] > 1:
        ax.errorbar(np.arange(len(top_n_sa_df.columns)), top_n_sa_df.mean(axis=0), yerr=top_n_sa_df.std(axis=0), fmt='o', capsize=5)
    ax.fill_between(np.arange(len(top_n_sa_df.columns)), top_n_sa_df.mean(axis=0) - top_n_sa_df.std(axis=0), top_n_sa_df.mean(axis=0) + top_n_sa_df.std(axis=0), alpha=0.2)
    ax.set_ylabel('Mean Spectral Angle $\pm \sigma$')
    ax.set_xlabel('Entry Ranked by Asc. Spectral Angle')
    ax.set_xticks(top_n_sa_df.columns.to_list())
    xlbls=ax.set_xticklabels(top_n_sa_df.columns.to_list(), rotation=60, horizontalalignment='right', verticalalignment='top')

    # add vertical gridlines    
    ax.grid(which='major', axis='x', color='gray', linestyle='-', linewidth=0.2)

    # add a top x-axis called Match Rank, with ticks aligned to the right
    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    ax2.set_xticks(np.arange(len(top_n_sa_df.columns)))
    ax2.set_xticklabels(np.arange(1, len(top_n_sa_df.columns)+1), horizontalalignment='right', verticalalignment='center', fontsize=7)
    x2lbls=ax2.set_xlabel('Match Rank')

    title = f'{material_name} Mean Spectral Angle Match'
    fig.suptitle(title)

    return fig, ax

In [ ]:
import seaborn as sns

In [ ]:
def plot_top3_matches(obs, ref_obs, ref_matcol, material_name, top3, ax: plt.Axes=None):
    if ax is None:
        fig, ax = plt.subplots()
    else:
        fig = ax.figure

    top3_hires_df = ref_matcol.main_df.loc[top3]
    refl_df_top3_hires = top3_hires_df[ref_matcol.wvls]

    top3_obs_df = obs.main_df[obs.main_df['Root Data ID'].isin(top3)]
    # set Root Data ID as index
    top3_obs_df = top3_obs_df.set_index('Root Data ID')
    refl_df_top3 = top3_obs_df[obs.wvls]

    # get reverse order of top3
    top3_r = top3[::-1]
        
    refl_df_top3 = refl_df_top3.loc[top3_r]
    refl_df_top3_hires = refl_df_top3_hires.loc[top3_r]

    minima = refl_df_top3.min(axis=1)
    maxima = refl_df_top3.max(axis=1)
    spec_range = maxima - minima
    max_range = (spec_range).max()

    # offset the reflectance
    padding =  max_range * 1/6
    spec_range[spec_range < padding] = padding
    offsets = - minima + (spec_range+padding).cumsum().shift(periods=1, fill_value = 0) + padding/2   

    # # apply offsets to each entry
    # offsets = refl_df_top3.max(axis=1).cumsum()

    refl_df_top3 = refl_df_top3.add(offsets, axis=0)
    refl_df_top3_hires = refl_df_top3_hires.add(offsets, axis=0)

    maxima = refl_df_top3_hires.max(axis=1)

    cmap = plt.get_cmap('tab10_r')

    refl_df_top3_hires.T.plot(ax=ax, linewidth=0.8)
    l = ax.get_lines()
    refl_df_top3.T.plot(ax=ax, color=[i.get_color() for i in l])
    
    # add padding to y lim
    # get ylim
    ylim = ax.get_ylim()
    ax.set_ylim(ylim[0], ylim[1]+padding)

    # despine
    sns.despine(ax=ax)

    # xlabel
    ax.set_xlabel('Wavelength (nm)')
    # ylabel
    ax.set_ylabel('Stacked Reflectance')

    # no legend
    ax.legend().set_visible(False)

    # annotate with the top3 at the end of each plot line
    for i, line in enumerate(l):
        ax.annotate(f'{3-i}. {top3_r[i]}', xy=(line.get_xdata()[0], maxima[top3_r[i]]+padding), color=line.get_color())

    ax.set_title(f'Top 3 {material_name} Spectral Angle Matches')

    return fig, ax

In [ ]:
def plot_spectral_angle_noisy_map(sa_df, material_name, top_n=10, ax: plt.Axes=None):
    if ax is None:
        fig, ax = plt.subplots()
    else:
        fig = ax.figure

    top_n_sa_df = sa_df.iloc[:,:top_n]

    im = ax.imshow(top_n_sa_df.to_numpy(), 
                cmap='viridis_r', 
                interpolation='none', resample=False,
                # vmin=0, vmax=1,
                aspect='auto')
    # add color bar called 'Spectral Angle'
    ax.set_xlabel('Entry Ranked by Asc. Baseline Spectral Angle')
    ax.set_ylabel('N-Repeat')
    cbar = ax.figure.colorbar(im, ax=ax, location='right')
    cbar.ax.set_ylabel('Spectral Angle')
    # use index for axis labels
    # set every tickmark to the corresponding index
    xticks=ax.set_xticks(np.arange(len(top_n_sa_df.columns))+0.5)
    xlbls=ax.set_xticklabels(top_n_sa_df.columns.to_list(), rotation=60, horizontalalignment='right', verticalalignment='top')

    # add vertical gridlines    
    ax.grid(which='major', axis='x', color='white', linestyle='-', linewidth=0.2)

    # add a top y-axis called Match Rank, with ticks aligned to the right
    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    ax2.set_xticks(np.arange(len(top_n_sa_df.columns))+0.5)
    ax2.set_xticklabels(np.arange(1, len(top_n_sa_df.columns)+1), horizontalalignment='right', verticalalignment='bottom', fontsize=7)
    ax2.set_xlabel('Match Rank')

    fig.suptitle(f'{material_name} Spectral Angle Match \n Enfys InAs MWIR SNR/10')
    fig.tight_layout()

    return fig, ax

In [ ]:
fig, ax = plot_spectral_angle_noisy_map(sa_df, mica_test_material)

Of course, errorbars only work if there are multiple repeat noisy samples.

In [ ]:
fig, ax = plot_mean_spectral_angle(sa_df, sa_bl, mica_test_material)

In [ ]:
top3 = sa_df.columns[:3].to_list()
top3

In [ ]:
# drop the full stop at the end of each if it is there
top3_d = [item.rstrip('.') for item in top3] 

In [ ]:
mica_ob.plot_profiles(stacked=True, scope=top3_d,hires_under=False)

In [ ]:
fig, ax = plot_top3_matches(mica_ob, mica_ob, mica_mc, mica_test_material, top3_d)

Now running it using the actual data:

In [ ]:
sa_df, sa_bl = compute_spectral_angle(obs, mica_ob, test_material)

In [ ]:
sa_df.mean(axis=0).sort_values()

In [ ]:
fig, ax = plot_spectral_angle_noisy_map(sa_df, test_material)

In [ ]:
fig, ax = plot_mean_spectral_angle(sa_df, sa_bl, test_material)

In [ ]:
top3 = sa_df.columns[:3].to_list()
top3

In [ ]:
top3_d = [item.rstrip('.') for item in top3] 

In [ ]:
mica_ob.plot_profiles(stacked=True, scope=top3_d,groupby='Species',hires_under=False)

In [ ]:
obs.plot_profiles(stacked=True, scope=test_material)

Now let's try this on the Continuum Removed spectra.

In [ ]:
mica_cr_tool = sla(mica_ob)
mica_cr_obs = mica_cr_tool.remove_continuum()

In [ ]:
cr_obs

In [ ]:
sa_cr_df, sa_cr_bl = compute_spectral_angle(cr_obs, mica_cr_obs, test_material)

In [ ]:
sa_cr_df = sa_cr_df.loc[:, sa_cr_df.mean(axis=0).sort_values().index]

In [ ]:
sa_cr_df

In [ ]:
fig, ax = plot_spectral_angle_noisy_map(sa_cr_df, test_material)

In [ ]:
fig, ax = plot_mean_spectral_angle(sa_cr_df, test_material)

In [ ]:
top3 = sa_cr_df.columns[:5].to_list()
top3

In [ ]:
top3_d = [item.rstrip('.') for item in top3] 

In [ ]:
mica_cr_obs.plot_profiles(stacked=True, scope=top3_d,groupby='Category',hires_under=False)

In [ ]:
cr_obs.plot_profiles(stacked=True, scope=test_material)

# Conclusions

# References